<a href="https://colab.research.google.com/github/laulivette/poc_indutech/blob/main/poc_indutech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Étape 1 — Configurez Redpanda


### 1.1 Installation de Redpanda

In [1]:
%%bash
# Installation de Redpanda (paquet officiel .deb)
curl -1sLf 'https://dl.redpanda.com/nzc4ZYQK3WRGd9sy/redpanda/cfg/setup/bash.deb.sh' | sudo -E bash > /dev/null 2>&1
sudo apt-get install redpanda -y -qq > /dev/null 2>&1
echo "Redpanda installe avec succes"
rpk version

Redpanda installe avec succes
rpk version: v26.2.1
Git ref:     8cd781be99c737c7d147d9dc239a0bbdd591a4d8
Build date:  2026 Jul 24 10 13 57 Fri
OS/Arch:     linux/amd64
Go version:  go1.26.5

Redpanda Cluster
  Unreachable, to debug, use the '-v' flag. To get the broker versions, pass the
  hosts via flags, profile, or environment variables:
    rpk version -X admin.hosts=<host address>

  To get only the rpk version, use 'rpk --version'.


### 1.2 Lancement de Redpanda

In [2]:
import subprocess, time

# Lancement Redpanda en arriere-plan (mode dev, un seul noeud)
proc = subprocess.Popen(
    ["sudo", "rpk", "redpanda", "start",
     "--mode", "dev-container",
     "--smp", "1",
     "--memory", "1G",
     "--overprovisioned",
     "--default-log-level=info"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(8)  # laisse le temps au cluster de demarrer
print("Redpanda demarre (PID:", proc.pid, ")")

Redpanda demarre (PID: 74290 )


In [3]:
# Verification que le cluster est operationnel
!rpk cluster info

CLUSTER
redpanda.1a4a16d8-a479-434a-b4ce-e5ba1516ea3e

BROKERS
ID    HOST       PORT
0*    127.0.0.1  9092

TOPICS
NAME            PARTITIONS  REPLICAS
client_tickets  3           1



### 1.3 Création du topic `client_tickets`

In [4]:
!rpk topic create client_tickets --partitions 3 --replicas 1
!rpk topic list

TOPIC           STATUS
client_tickets  TOPIC_ALREADY_EXISTS: The topic has already been created
NAME            PARTITIONS  REPLICAS
client_tickets  3           1


### 1.4 Script producteur — génération de tickets aléatoires

On installe `confluent-kafka` (compatible avec l'API Kafka exposée par Redpanda) puis on écrit un générateur de tickets.

In [5]:
!pip install confluent-kafka faker -q

In [6]:
import json
import random
import uuid
from datetime import datetime, timezone
from confluent_kafka import Producer
from faker import Faker

fake = Faker("fr_FR")

TOPIC = "client_tickets"
BOOTSTRAP_SERVERS = "localhost:9092"

TYPES_DEMANDE = [
    "Probleme technique",
    "Facturation",
    "Demande d'information",
    "Reclamation",
    "Resiliation",
    "Support produit",
]

PRIORITES = ["Basse", "Moyenne", "Haute", "Critique"]

DEMANDES_TEMPLATES = {
    "Probleme technique": "Le service ne repond plus depuis ce matin.",
    "Facturation": "Le montant preleve ne correspond pas au contrat.",
    "Demande d'information": "Je souhaite en savoir plus sur l'offre premium.",
    "Reclamation": "Le delai de livraison n'a pas ete respecte.",
    "Resiliation": "Je souhaite resilier mon abonnement.",
    "Support produit": "Je n'arrive pas a configurer le produit.",
}

def generer_ticket():
    type_demande = random.choice(TYPES_DEMANDE)
    return {
        "ticket_id": str(uuid.uuid4()),
        "client_id": fake.uuid4(),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "demande": DEMANDES_TEMPLATES[type_demande],
        "type_demande": type_demande,
        "priorite": random.choice(PRIORITES),
    }

def delivery_report(err, msg):
    if err is not None:
        print(f"Echec de livraison: {err}")

producer = Producer({"bootstrap.servers": BOOTSTRAP_SERVERS})

print("Envoi des tickets vers le topic", TOPIC)


Envoi des tickets vers le topic client_tickets


In [7]:
import time

NB_TICKETS = 200          # nombre de tickets a generer
INTERVALLE_SEC = 0.05      # delai entre deux envois (simulateur temps reel)

for i in range(NB_TICKETS):
    ticket = generer_ticket()
    producer.produce(
        TOPIC,
        key=ticket["ticket_id"],
        value=json.dumps(ticket).encode("utf-8"),
        callback=delivery_report,
    )
    producer.poll(0)
    time.sleep(INTERVALLE_SEC)

producer.flush()
print(f"{NB_TICKETS} tickets envoyes dans le topic '{TOPIC}'")


200 tickets envoyes dans le topic 'client_tickets'


In [8]:
# Verification rapide : consommation de quelques messages depuis le topic
!rpk topic consume client_tickets --num 3 --format json

{
  "topic": "client_tickets",
  "key": "78f12a1c-3a3e-47a0-9db3-41732db983f4",
  "value": "{\"ticket_id\": \"78f12a1c-3a3e-47a0-9db3-41732db983f4\", \"client_id\": \"54deb28b-ba01-43ea-9487-7780d22f2d6b\", \"created_at\": \"2026-08-05T07:42:33.428171+00:00\", \"demande\": \"Le service ne repond plus depuis ce matin.\", \"type_demande\": \"Probleme technique\", \"priorite\": \"Basse\"}",
  "timestamp": 1785915753428,
  "partition": 2,
  "offset": 0
}
{
  "topic": "client_tickets",
  "key": "bca4db22-c80d-48a1-b11b-b9853f4c6e27",
  "value": "{\"ticket_id\": \"bca4db22-c80d-48a1-b11b-b9853f4c6e27\", \"client_id\": \"99cb8509-31b7-46e6-833d-25ce71f78d9f\", \"created_at\": \"2026-08-05T07:42:33.529251+00:00\", \"demande\": \"Le montant preleve ne correspond pas au contrat.\", \"type_demande\": \"Facturation\", \"priorite\": \"Haute\"}",
  "timestamp": 1785915753529,
  "partition": 2,
  "offset": 1
}
{
  "topic": "client_tickets",
  "key": "8f748a62-4287-4b3c-8c7e-edece5e13542",
  "value": 

## Étape 2 — Traitez les données avec PySpark

### 2.1 Installation de Java + PySpark

In [9]:
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.1 -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
print("Java et PySpark installes")

Java et PySpark installes


### 2.2 Création de la session Spark avec le connecteur Kafka

In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ClientTicketsProcessing")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .config("spark.sql.shuffle.partitions", "4")   # adapte a la taille du POC
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Session Spark prete :", spark.version)

Session Spark prete : 3.5.1


### 2.3 Lecture des données (batch) depuis le topic Redpanda

In [11]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, from_json, count as spark_count, when

schema = StructType([
    StructField("ticket_id", StringType()),
    StructField("client_id", StringType()),
    StructField("created_at", StringType()),
    StructField("demande", StringType()),
    StructField("type_demande", StringType()),
    StructField("priorite", StringType()),
])

df_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "client_tickets")
    .option("startingOffsets", "earliest")
    .load()
)

df_tickets = (
    df_raw
    .select(from_json(col("value").cast("string"), schema).alias("data"))
    .select("data.*")
)

df_tickets.show(5, truncate=False)
print("Nombre de tickets lus :", df_tickets.count())


+------------------------------------+------------------------------------+--------------------------------+------------------------------------------------+------------------+--------+
|ticket_id                           |client_id                           |created_at                      |demande                                         |type_demande      |priorite|
+------------------------------------+------------------------------------+--------------------------------+------------------------------------------------+------------------+--------+
|bd477ccc-d7ac-4167-b417-2853354425bb|7737931a-7170-413b-87f5-da264aaa1f3e|2026-08-05T07:42:33.377645+00:00|Le montant preleve ne correspond pas au contrat.|Facturation       |Moyenne |
|fd3293a8-94d7-4d9c-8279-808fde03653b|a284f14c-3d7c-40a2-8a7d-58944394d1d0|2026-08-05T07:42:33.478694+00:00|Je n'arrive pas a configurer le produit.        |Support produit   |Moyenne |
|43c96f0a-23e0-40c2-b201-15d67037b421|c53a7214-7607-4997-a8cd-1d92edc1

### 2.4 Transformations et agrégations

In [12]:
# Mapping type de demande -> equipe de support assignee
df_enrichi = df_tickets.withColumn(
    "equipe_support",
    when(col("type_demande") == "Probleme technique", "Equipe Technique")
    .when(col("type_demande") == "Facturation", "Equipe Facturation")
    .when(col("type_demande") == "Demande d'information", "Equipe Relation Client")
    .when(col("type_demande") == "Reclamation", "Equipe Qualite")
    .when(col("type_demande") == "Resiliation", "Equipe Retention")
    .when(col("type_demande") == "Support produit", "Equipe Support Produit")
    .otherwise("Equipe Generale")
)

df_enrichi.show(5, truncate=False)

+------------------------------------+------------------------------------+--------------------------------+------------------------------------------------+------------------+--------+----------------------+
|ticket_id                           |client_id                           |created_at                      |demande                                         |type_demande      |priorite|equipe_support        |
+------------------------------------+------------------------------------+--------------------------------+------------------------------------------------+------------------+--------+----------------------+
|bd477ccc-d7ac-4167-b417-2853354425bb|7737931a-7170-413b-87f5-da264aaa1f3e|2026-08-05T07:42:33.377645+00:00|Le montant preleve ne correspond pas au contrat.|Facturation       |Moyenne |Equipe Facturation    |
|fd3293a8-94d7-4d9c-8279-808fde03653b|a284f14c-3d7c-40a2-8a7d-58944394d1d0|2026-08-05T07:42:33.478694+00:00|Je n'arrive pas a configurer le produit.        |Support

In [13]:
# Agregation : nombre de tickets par type de demande
tickets_par_type = (
    df_enrichi.groupBy("type_demande")
    .agg(spark_count("*").alias("nb_tickets"))
    .orderBy(col("nb_tickets").desc())
)
tickets_par_type.show()

+--------------------+----------+
|        type_demande|nb_tickets|
+--------------------+----------+
|Demande d'informa...|       188|
|     Support produit|       178|
|  Probleme technique|       162|
|         Facturation|       161|
|         Reclamation|       160|
|         Resiliation|       151|
+--------------------+----------+



In [14]:
# Agregation : nombre de tickets par equipe et par priorite
tickets_par_equipe_priorite = (
    df_enrichi.groupBy("equipe_support", "priorite")
    .agg(spark_count("*").alias("nb_tickets"))
    .orderBy(col("equipe_support"), col("priorite"))
)
tickets_par_equipe_priorite.show(20)

+--------------------+--------+----------+
|      equipe_support|priorite|nb_tickets|
+--------------------+--------+----------+
|  Equipe Facturation|   Basse|        40|
|  Equipe Facturation|Critique|        39|
|  Equipe Facturation|   Haute|        48|
|  Equipe Facturation| Moyenne|        34|
|      Equipe Qualite|   Basse|        38|
|      Equipe Qualite|Critique|        46|
|      Equipe Qualite|   Haute|        36|
|      Equipe Qualite| Moyenne|        40|
|Equipe Relation C...|   Basse|        65|
|Equipe Relation C...|Critique|        30|
|Equipe Relation C...|   Haute|        45|
|Equipe Relation C...| Moyenne|        48|
|    Equipe Retention|   Basse|        39|
|    Equipe Retention|Critique|        42|
|    Equipe Retention|   Haute|        38|
|    Equipe Retention| Moyenne|        32|
|Equipe Support Pr...|   Basse|        40|
|Equipe Support Pr...|Critique|        32|
|Equipe Support Pr...|   Haute|        46|
|Equipe Support Pr...| Moyenne|        60|
+----------

### 2.5 Gestion des erreurs / résilience (points de vigilance)

In [15]:
import time

def lire_depuis_kafka_avec_retry(spark, topic, max_retries=3, delay=5):
    for tentative in range(1, max_retries + 1):
        try:
            return (
                spark.read
                .format("kafka")
                .option("kafka.bootstrap.servers", "localhost:9092")
                .option("subscribe", topic)
                .option("startingOffsets", "earliest")
                .load()
            )
        except Exception as e:
            print(f"Tentative {tentative}/{max_retries} echouee : {e}")
            if tentative == max_retries:
                raise
            time.sleep(delay)

# Exemple d'utilisation :
# df_raw = lire_depuis_kafka_avec_retry(spark, "client_tickets")

## Étape 3 — Exportez les données

In [16]:
import os

OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Export JSON
df_enrichi.coalesce(1).write.mode("overwrite").json(f"{OUTPUT_DIR}/tickets_enrichis_json")
tickets_par_type.coalesce(1).write.mode("overwrite").json(f"{OUTPUT_DIR}/tickets_par_type_json")

# Export Parquet
df_enrichi.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/tickets_enrichis_parquet")
tickets_par_equipe_priorite.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/tickets_par_equipe_priorite_parquet")

print("Exports termines dans", OUTPUT_DIR)
!ls -R {OUTPUT_DIR}

Exports termines dans /content/output
/content/output:
tickets_enrichis_json	  tickets_par_equipe_priorite_parquet
tickets_enrichis_parquet  tickets_par_type_json

/content/output/tickets_enrichis_json:
part-00000-f435f3b3-d023-4fd5-ba7a-6b97c4fe8107-c000.json  _SUCCESS

/content/output/tickets_enrichis_parquet:
part-00000-3b2fbca9-7ddc-4c52-9906-8cc15f037658-c000.snappy.parquet
part-00001-3b2fbca9-7ddc-4c52-9906-8cc15f037658-c000.snappy.parquet
part-00002-3b2fbca9-7ddc-4c52-9906-8cc15f037658-c000.snappy.parquet
_SUCCESS

/content/output/tickets_par_equipe_priorite_parquet:
part-00000-4b594545-8f26-485e-80b7-58265ab5d617-c000.snappy.parquet  _SUCCESS

/content/output/tickets_par_type_json:
part-00000-220a4351-6eb6-4f6b-9ee2-ee6a0cebd05d-c000.json  _SUCCESS


In [17]:
# Sauvegarde persistante sur Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DEST = "/content/drive/MyDrive/poc_redpanda_pyspark_output"
shutil.copytree(OUTPUT_DIR, DRIVE_DEST, dirs_exist_ok=True)
print("Copie terminee vers :", DRIVE_DEST)


MessageError: Error: credential propagation was unsuccessful